# FashionSense AI — Advanced Training Notebook
**ViT-B/16 Fine-tuning + Siamese Compatibility | Full EDA + Visualizations**

### Before running:
1. Settings → Add data:
   - `lyy1994/deepfashion2` **(required)**
   - `zalando-research/fashionmnist` **(required)**
   - Polyvore **(optional)** — search Kaggle for `polyvore`. If not found, Phase 2 automatically falls back to **SyntheticCompatibilityDataset** built from DeepFashion2 using fashion body-zone rules. No action needed.
2. Settings → Accelerator → **GPU T4 x2** or P100
3. Run All

**Sections:**
1. Environment & Config
2. Exploratory Data Analysis (EDA)
3. Model Architecture Diagram
4. Dataset Classes & DataLoaders
5. Phase 1 — Classification Training
6. Phase 1 Evaluation (ROC, Confusion Matrix, t-SNE)
7. Phase 2 — Compatibility Training
8. Grad-CAM Gallery
9. Final Dashboard

## 1. Environment Setup & Config

In [ ]:
!pip install -q transformers==4.40.0 open_clip_torch umap-learn

In [ ]:
import os, sys, json, math, random, warnings, time, logging
from pathlib import Path
from collections import Counter, defaultdict
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from transformers import CLIPModel, CLIPTokenizer

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Dataset base paths ────────────────────────────────────────────────────────
DF2_ROOT    = Path('/kaggle/input/datasets/thusharanair/deepfashion2-original-with-dataframes')
FMNIST_ROOT = Path('/kaggle/input/datasets/organizations/zalando-research/fashionmnist')
OUT_DIR     = Path('/kaggle/working')
CKPT_DIR    = OUT_DIR / 'checkpoints'
VIZ_DIR     = OUT_DIR / 'visualizations'
for d in [CKPT_DIR, VIZ_DIR]: d.mkdir(parents=True, exist_ok=True)

# ── DeepFashion2 exact sub-paths (confirmed from dataset panel) ───────────────
DF2_IMG_ROOT = DF2_ROOT / 'DeepFashion2' / 'deepfashion2_original_images'
DF2_CSV_ROOT = DF2_ROOT / 'DeepFashion2' / 'img_info_dataframes'
DF2_ANNO_MODE = 'csv'   # this dataset uses CSV dataframes, not per-image JSONs

# ── Polyvore auto-detection ───────────────────────────────────────────────────
POLYVORE_ROOT = None
for _slug in ['polyvore-outfits', 'polyvore', 'polyvore-dataset',
              'polyvore-outfits-dataset', 'fashion-compatibility-polyvore']:
    _p = Path(f'/kaggle/input/{_slug}')
    if _p.exists():
        POLYVORE_ROOT = _p
        break
if POLYVORE_ROOT is None:
    for _d in Path('/kaggle/input').iterdir():
        if (_d / 'images').exists() and list(_d.glob('*.json')):
            POLYVORE_ROOT = _d
            break

COMPAT_MODE = 'polyvore' if POLYVORE_ROOT else 'synthetic'

# ── Verify paths ──────────────────────────────────────────────────────────────
print(f'{"✓" if DF2_IMG_ROOT.exists() else "✗ NOT FOUND"}  DF2 images : {DF2_IMG_ROOT}')
print(f'{"✓" if DF2_CSV_ROOT.exists() else "✗ NOT FOUND"}  DF2 CSVs   : {DF2_CSV_ROOT}')
print(f'{"✓" if FMNIST_ROOT.exists()  else "✗ NOT FOUND"}  FMNIST     : {FMNIST_ROOT}')
if POLYVORE_ROOT:
    print(f'✓  Polyvore  : {POLYVORE_ROOT}')
else:
    print('⚠  Polyvore  : NOT FOUND — using SyntheticCompatibilityDataset')
print(f'\nAnnotation mode  : [{DF2_ANNO_MODE.upper()}]')
print(f'Compatibility mode: [{COMPAT_MODE.upper()}]')

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    clip_model     = 'openai/clip-vit-base-patch16',
    num_classes    = 13,
    emb_dim        = 512,
    image_size     = 224,
    # ── Phase 1: full fine-tuning with LLRD + warmup for 95%+ accuracy ───────
    lr_cls         = 3e-5,   # head LR; backbone gets decayed fraction via LLRD
    lr_decay       = 0.75,   # LLRD decay rate per ViT block depth
    warmup_epochs  = 3,      # linear warmup before cosine decay
    epochs_cls     = 30,     # 30 epochs gives model time to converge to 95%+
    bs_cls         = 64,
    mixup_alpha    = 0.1,    # reduced from 0.2 — milder MixUp for higher hard accuracy
    # ── Phase 2 ───────────────────────────────────────────────────────────────
    lr_compat      = 1e-5,
    epochs_compat  = 10,
    bs_compat      = 32,
    # ── Shared ────────────────────────────────────────────────────────────────
    weight_decay   = 0.01,
    grad_clip      = 1.0,
    triplet_margin = 0.3,
    num_workers    = 2,
    df2_max_train  = None,
    log_every      = 100,
)
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)
DF2_CATS  = [
    'Short Sleeve Top','Long Sleeve Top','Short Sleeve Outwear',
    'Long Sleeve Outwear','Vest','Sling','Shorts','Trousers',
    'Skirt','Short Sleeve Dress','Long Sleeve Dress','Vest Dress','Sling Dress',
]
DF2_ID_TO_IDX = {i+1: i for i in range(13)}
CAT_COLORS = plt.cm.tab20(np.linspace(0, 1, 13))
print(f'\nConfig loaded. {CFG["num_classes"]} categories.')

In [ ]:
# ── Dataset Structure Verification & CSV Column Preview ──────────────────────
# Run this cell immediately after Cell 4 to confirm everything is in order
# before any training code runs.

print('=' * 60)
print('DEEPFASHION2 — Image directories')
print('=' * 60)
for split in ['train', 'validation', 'test']:
    img_dir = DF2_IMG_ROOT / split
    if img_dir.exists():
        n_jpg = len(list(img_dir.glob('*.jpg')))
        n_png = len(list(img_dir.glob('*.png')))
        # Images may sit one level deeper (e.g. train/image/)
        if n_jpg + n_png == 0:
            sub = next((d for d in img_dir.iterdir() if d.is_dir()), None)
            if sub:
                n_jpg = len(list(sub.glob('*.jpg')))
                print(f'  ✓ {split}/  →  {split}/{sub.name}/  ({n_jpg:,} images)')
            else:
                print(f'  ⚠ {split}/  — 0 images found, check subdirectory')
        else:
            print(f'  ✓ {split}/  ({n_jpg + n_png:,} images)')
    else:
        print(f'  ✗ {split}/  NOT FOUND')

print()
print('=' * 60)
print('DEEPFASHION2 — CSV annotations (first 5 rows + columns)')
print('=' * 60)
for split in ['train', 'validation']:
    csv_path = DF2_CSV_ROOT / f'{split}.csv'
    if csv_path.exists():
        df_preview = pd.read_csv(csv_path, nrows=5)
        print(f'\n{split}.csv  ({csv_path.stat().st_size/1e6:.1f} MB)')
        print(f'  Columns : {list(df_preview.columns)}')
        print(f'  Shape   : {pd.read_csv(csv_path).shape}')
        print(df_preview.head(3).to_string(index=False))
    else:
        print(f'  ✗ {split}.csv  NOT FOUND at {csv_path}')

print()
print('=' * 60)
print('FASHION-MNIST')
print('=' * 60)
for fname in ['fashion-mnist_train.csv', 'fashion-mnist_test.csv']:
    p = FMNIST_ROOT / fname
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f'  ✓ {fname}  ({size_mb:.0f} MB)')
    else:
        print(f'  ✗ {fname}  NOT FOUND')

In [ ]:
# ── 2A. DeepFashion2 — Category Distribution (from CSV) ──────────────────────
print('Reading DeepFashion2 category distribution from CSVs...')

df2_splits = {}
for split in ['train', 'validation']:
    csv_path = DF2_CSV_ROOT / f'{split}.csv'
    df = pd.read_csv(csv_path)

    # Auto-detect the category column
    cat_col = next((c for c in df.columns
                    if c.lower() in ('category_id','cat_id','label','category','class_id')), None)
    if cat_col is None:
        print(f'  [WARN] No category column found in {split}.csv — columns: {list(df.columns)}')
        continue

    counts = Counter()
    for cat_id, grp in df.groupby(cat_col):
        idx = DF2_ID_TO_IDX.get(int(cat_id))
        if idx is not None:
            counts[DF2_CATS[idx]] = len(grp)
    df2_splits[split] = counts
    print(f'  {split}: {sum(counts.values()):,} labelled samples across {len(counts)} categories')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (split, counts) in zip(axes, df2_splits.items()):
    cats  = list(counts.keys())
    vals  = list(counts.values())
    order = np.argsort(vals)[::-1]
    bars  = ax.barh([cats[i] for i in order], [vals[i] for i in order],
                    color=CAT_COLORS[[DF2_CATS.index(cats[i]) for i in order]])
    ax.set_xlabel('Sample Count')
    ax.set_title(f'DeepFashion2 — {split.capitalize()} Category Distribution', fontsize=13)
    for bar, val in zip(bars, [vals[i] for i in order]):
        ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)

plt.suptitle('DeepFashion2 Class Balance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'df2_class_distribution.png')
plt.show()
print('Done.')

In [ ]:
# -- 2B. DeepFashion2 Sample Image Grid (confirmed columns: path, category_id)
df_train = pd.read_csv(DF2_CSV_ROOT / 'train.csv', usecols=['path', 'category_id'])
print(f'train.csv rows: {len(df_train):,}')
print(f'Sample path   : {df_train["path"].iloc[0]}')

def _find_split_img_dir(split):
    p = DF2_IMG_ROOT / split / 'image'
    if p.exists(): return p
    base = DF2_IMG_ROOT / split
    for sub in (sorted(base.iterdir()) if base.exists() else []):
        if sub.is_dir() and any(sub.glob('*.jpg')): return sub
    return DF2_IMG_ROOT / split

img_dir = _find_split_img_dir('train')
print(f'Image dir: {img_dir}  exists={img_dir.exists()}')

cat_to_sample = {}
for row in df_train.itertuples(index=False):
    cat_id = int(row.category_id)
    idx    = DF2_ID_TO_IDX.get(cat_id)
    if idx is None or idx in cat_to_sample: continue
    p = img_dir / Path(str(row.path)).name
    if p.exists(): cat_to_sample[idx] = (DF2_CATS[idx], p)
    if len(cat_to_sample) == 13: break

print(f'Found samples for {len(cat_to_sample)}/13 categories')

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flat
for i, (idx, (cat, path)) in enumerate(sorted(cat_to_sample.items())):
    img = Image.open(path).convert('RGB'); img.thumbnail((200, 250))
    axes[i].imshow(img)
    axes[i].set_title(cat, fontsize=9, fontweight='bold', color='white', pad=3,
                      bbox=dict(boxstyle='round,pad=0.2', facecolor=CAT_COLORS[idx], alpha=0.85))
    axes[i].axis('off')
for j in range(len(cat_to_sample), len(axes)): axes[j].axis('off')
plt.suptitle('DeepFashion2 -- One Sample per Category', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / 'df2_category_samples.png', bbox_inches='tight')
plt.show()
sample_img_path = list(cat_to_sample.values())[0][1] if cat_to_sample else next(img_dir.glob('*.jpg'))


In [ ]:
# -- 2C. Image Resolution from CSV (img_width/img_height columns — no image opening)
df_dims = pd.read_csv(DF2_CSV_ROOT / 'train.csv', usecols=['img_width', 'img_height']).dropna()
widths  = df_dims['img_width'].astype(int).tolist()
heights = df_dims['img_height'].astype(int).tolist()
print(f'Dimension rows: {len(widths):,}')

if not widths:
    print('[WARN] No dimension data')
else:
    med_w, med_h = int(np.median(widths)), int(np.median(heights))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(widths,  bins=50, color='steelblue', alpha=0.8, edgecolor='white')
    axes[0].axvline(med_w, color='red',  linestyle='--', label=f'Median {med_w}px')
    axes[0].set_title('Image Width Distribution'); axes[0].set_xlabel('Width (px)'); axes[0].legend()
    axes[1].hist(heights, bins=50, color='coral', alpha=0.8, edgecolor='white')
    axes[1].axvline(med_h, color='navy', linestyle='--', label=f'Median {med_h}px')
    axes[1].set_title('Image Height Distribution'); axes[1].set_xlabel('Height (px)'); axes[1].legend()
    axes[2].scatter(widths[:5000], heights[:5000], alpha=0.2, s=4, color='mediumpurple')
    axes[2].axvline(224, color='red', linestyle='--', alpha=0.7, label='224px (model input)')
    axes[2].axhline(224, color='red', linestyle='--', alpha=0.7)
    axes[2].set_title('Width vs Height (5K sample)'); axes[2].set_xlabel('Width'); axes[2].set_ylabel('Height'); axes[2].legend()
    plt.suptitle(f'DeepFashion2 Resolution (n={len(widths):,} from CSV)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.savefig(VIZ_DIR / 'df2_image_dimensions.png'); plt.show()


In [ ]:
# ── 2D. Fashion-MNIST — Class Distribution + Sample Grid ────────────────────
fmnist_df = pd.read_csv(FMNIST_ROOT / 'fashion-mnist_train.csv')
FMNIST_LABELS = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
                 'Sandal','Shirt','Sneaker','Bag','Ankle boot']

fig = plt.figure(figsize=(20, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1.2, 1])

# Bar chart
ax_bar = fig.add_subplot(gs[:, 0])
counts = fmnist_df['label'].value_counts().sort_index()
bars = ax_bar.bar(FMNIST_LABELS, counts.values, color=plt.cm.Set2(np.linspace(0,1,10)), edgecolor='white')
ax_bar.set_xticklabels(FMNIST_LABELS, rotation=35, ha='right')
ax_bar.set_ylabel('Sample Count'); ax_bar.set_title('Fashion-MNIST — Class Distribution (60K train)', fontsize=13)
for bar in bars:
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{int(bar.get_height()):,}', ha='center', fontsize=9)

# Sample grid
ax_grid = fig.add_subplot(gs[:, 1])
ax_grid.axis('off')
inner = gridspec.GridSpecFromSubplotSpec(2, 5, subplot_spec=gs[:, 1], hspace=0.05, wspace=0.05)
for cls_id in range(10):
    row_data = fmnist_df[fmnist_df['label'] == cls_id].iloc[0]
    pixels = row_data.drop('label').values.reshape(28, 28).astype(np.uint8)
    ax = fig.add_subplot(inner[cls_id // 5, cls_id % 5])
    ax.imshow(pixels, cmap='gray_r')
    ax.set_title(FMNIST_LABELS[cls_id], fontsize=7)
    ax.axis('off')

plt.suptitle('Fashion-MNIST Overview', fontsize=15, fontweight='bold')
plt.savefig(VIZ_DIR / 'fmnist_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2E. Fashion-MNIST — Pixel Intensity Distribution per Class ───────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharey=True)
for cls_id, ax in enumerate(axes.flat):
    class_pixels = fmnist_df[fmnist_df['label'] == cls_id].drop(columns='label').values.flatten()
    ax.hist(class_pixels, bins=50, color=plt.cm.Set2(cls_id / 10), alpha=0.85, edgecolor='none')
    ax.set_title(FMNIST_LABELS[cls_id], fontsize=9)
    ax.axvline(class_pixels.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'μ={class_pixels.mean():.0f}')
    ax.legend(fontsize=7)
    ax.set_xlabel('Pixel value')

plt.suptitle('Fashion-MNIST — Pixel Intensity Distribution per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'fmnist_pixel_distribution.png')
plt.show()

In [ ]:
# ── 2F. Dataset Statistics Summary Table ─────────────────────────────────────
if POLYVORE_ROOT:
    compat_name  = f'Polyvore ({POLYVORE_ROOT.name})'
    compat_train = '~17K outfits'
    compat_val   = '~4K outfits'
    compat_size  = 'Variable'
    compat_fmt   = 'JPEG + JSON'
else:
    compat_name  = 'Synthetic (DeepFashion2-derived)'
    compat_train = 'Derived from DF2 train'
    compat_val   = 'Derived from DF2 val'
    compat_size  = 'Same as DeepFashion2'
    compat_fmt   = 'Rule-based triplets'

stats = {
    'Dataset':    ['DeepFashion2', 'Fashion-MNIST', compat_name],
    'Role':       ['Primary Classification', 'Warm-up / Baseline', 'Compatibility Training'],
    'Train Size': ['~442K', '60,000', compat_train],
    'Val/Test':   ['~49K',  '10,000', compat_val],
    'Classes':    [13, 10, 'N/A (triplets)'],
    'Image Size': ['Variable (800×1000 avg)', '28×28 → 224×224', compat_size],
    'Format':     ['JPEG + JSON anno', 'CSV (pixel rows)', compat_fmt],
}
df_stats = pd.DataFrame(stats).set_index('Dataset')

fig, ax = plt.subplots(figsize=(15, 3))
ax.axis('off')
table = ax.table(
    cellText  = df_stats.values,
    colLabels = df_stats.columns,
    rowLabels = df_stats.index,
    cellLoc   = 'center',
    loc       = 'center',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)
for (r, c), cell in table.get_celld().items():
    if r == 0 or c == -1:
        cell.set_facecolor('#2D3A4A')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F0F4F8')

plt.title(f'Dataset Summary  |  Compatibility mode: [{COMPAT_MODE.upper()}]',
          fontsize=14, fontweight='bold', pad=12)
plt.savefig(VIZ_DIR / 'dataset_summary_table.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2G. Data Augmentation Pipeline Visualization ─────────────────────────────
# Show how a single image changes through each augmentation step
sample_img_path = next((DF2_IMG_ROOT / 'train' / 'image').glob('*.jpg'))
orig_img = Image.open(sample_img_path).convert('RGB')

aug_steps = [
    ('Original',           T.Compose([T.Resize((224, 224))])),
    ('RandomResizedCrop',  T.Compose([T.RandomResizedCrop(224, scale=(0.5, 0.7))])),
    ('HorizontalFlip',     T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(p=1.0)])),
    ('ColorJitter',        T.Compose([T.Resize((224,224)), T.ColorJitter(0.5, 0.5, 0.5, 0.2)])),
    ('Grayscale',          T.Compose([T.Resize((224,224)), T.Grayscale(3)])),
    ('Final (normalized)', T.Compose([T.Resize((224,224)), T.ToTensor(),
                                      T.Normalize(CLIP_MEAN, CLIP_STD)])),
]

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, (name, tfm) in zip(axes, aug_steps):
    result = tfm(orig_img)
    if isinstance(result, torch.Tensor):
        # Denormalize for display
        mean = torch.tensor(CLIP_MEAN).view(3,1,1)
        std  = torch.tensor(CLIP_STD).view(3,1,1)
        result = (result * std + mean).clamp(0,1).permute(1,2,0).numpy()
    else:
        result = np.array(result)
    ax.imshow(result)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Augmentation Pipeline — Step by Step', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'augmentation_pipeline.png', bbox_inches='tight')
plt.show()

## 3. Model Architecture Diagram

In [ ]:
# ── 3A. Frozen vs Trainable Layers Diagram ───────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 10); ax.set_ylim(0, 16); ax.axis('off')
ax.set_facecolor('#1A1A2E')
fig.patch.set_facecolor('#1A1A2E')

FROZEN    = '#4A5568'
TRAINABLE = '#48BB78'
HEAD_CLR  = '#ED8936'
ARROW_CLR = '#A0AEC0'

def draw_block(ax, x, y, w, h, color, label, sublabel='', alpha=0.9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                                    facecolor=color, edgecolor='white', linewidth=1.2, alpha=alpha)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2 + (0.12 if sublabel else 0), label,
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if sublabel:
        ax.text(x+w/2, y+h/2 - 0.15, sublabel, ha='center', va='center', fontsize=7, color='#E2E8F0')

# Input
draw_block(ax, 3.5, 14.0, 3, 0.7, '#2B6CB0', 'Input Image', '224×224×3')
ax.annotate('', xy=(5, 13.5), xytext=(5, 14.0),
            arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1.5))

# Patch embed
draw_block(ax, 3.5, 12.7, 3, 0.7, FROZEN, 'Patch Embedding', '14×14 patches → 196 tokens')
ax.annotate('', xy=(5, 12.2), xytext=(5, 12.7),
            arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1.5))

# ViT blocks — frozen (8)
for i in range(8):
    y = 11.5 - i * 0.8
    draw_block(ax, 2.8, y, 4.4, 0.65, FROZEN,
               f'Transformer Block {i+1}', 'FROZEN', alpha=0.7)
    if i < 7:
        ax.annotate('', xy=(5, y), xytext=(5, y+0.65),
                    arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1))

ax.annotate('', xy=(5, 5.3), xytext=(5, 5.15+0.65),
            arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1.5))

# ViT blocks — trainable (last 4)
for i in range(4):
    y = 4.8 - i * 0.9
    draw_block(ax, 2.8, y, 4.4, 0.75, TRAINABLE,
               f'Transformer Block {i+9}', 'TRAINABLE ✓')
    if i < 3:
        ax.annotate('', xy=(5, y), xytext=(5, y+0.75),
                    arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1.5))

ax.annotate('', xy=(5, 1.0), xytext=(5, 1.2),
            arrowprops=dict(arrowstyle='->', color=ARROW_CLR, lw=1.5))

# Projection + classifier head
draw_block(ax, 3.0, 0.3, 1.7, 0.65, TRAINABLE, 'Linear Proj', '768→512 emb')
draw_block(ax, 5.3, 0.3, 1.7, 0.65, HEAD_CLR,  'Classifier',  '512→256→13')

# Legend
legend_elements = [
    mpatches.Patch(facecolor=FROZEN,    label='Frozen (no gradient)'),
    mpatches.Patch(facecolor=TRAINABLE, label='Trainable (gradient flows)'),
    mpatches.Patch(facecolor=HEAD_CLR,  label='Classification head (new)'),
]
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.3,
          labelcolor='white', fontsize=9)

ax.set_title('FashionSense AI — ViT-B/16 Architecture\n(Frozen: 8 blocks | Trainable: 4 blocks + head)',
             color='white', fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(VIZ_DIR / 'model_architecture.png', bbox_inches='tight', facecolor='#1A1A2E')
plt.show()

In [ ]:
# ── 3B. Parameter Count Breakdown ────────────────────────────────────────────
# Load model to count params
class FashionVisualEncoder(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        clip = CLIPModel.from_pretrained(CFG['clip_model'])
        self.vision_model      = clip.vision_model
        self.visual_projection = clip.visual_projection
        layers = self.vision_model.encoder.layers
        for i, layer in enumerate(layers):
            for p in layer.parameters():
                p.requires_grad = (i >= len(layers) - 4)
        for p in self.vision_model.embeddings.parameters(): p.requires_grad = False
        for p in self.vision_model.pre_layrnorm.parameters(): p.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(512), nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        out = self.vision_model(pixel_values=x)
        emb = F.normalize(self.visual_projection(out.pooler_output), dim=-1)
        return emb, self.classifier(emb)

class CompatibilityModel(nn.Module):
    def __init__(self, input_dim=512, proj_dim=128):
        super().__init__()
        self.head = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, proj_dim))
        self.scorer = nn.Sequential(nn.Linear(proj_dim*2, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
    def project(self, x): return F.normalize(self.head(x), dim=-1)
    def score(self, a, b): return self.scorer(torch.cat([self.project(a), self.project(b)], -1)).squeeze(-1)*100
    def forward(self, a, p, n): return self.project(a), self.project(p), self.project(n)

class FashionSenseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_encoder = FashionVisualEncoder()
        self.compatibility  = CompatibilityModel()
    def forward_classification(self, x): return self.visual_encoder(x)
    def forward_triplet(self, a, p, n):
        ea,_ = self.visual_encoder(a); ep,_ = self.visual_encoder(p); en,_ = self.visual_encoder(n)
        return self.compatibility(ea, ep, en)

model = FashionSenseModel().to(DEVICE)

# Count params per component
def count_params(module):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

components = {
    'Frozen ViT blocks (1-8)':  sum(p.numel() for i, l in enumerate(model.visual_encoder.vision_model.encoder.layers) for p in l.parameters() if i < 8),
    'Trainable ViT blocks (9-12)': sum(p.numel() for i, l in enumerate(model.visual_encoder.vision_model.encoder.layers) for p in l.parameters() if i >= 8),
    'Patch Embedding':           sum(p.numel() for p in model.visual_encoder.vision_model.embeddings.parameters()),
    'Visual Projection':         sum(p.numel() for p in model.visual_encoder.visual_projection.parameters()),
    'Classifier Head':           sum(p.numel() for p in model.visual_encoder.classifier.parameters()),
    'Compatibility Module':      sum(p.numel() for p in model.compatibility.parameters()),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
colors_pie = ['#4A5568','#48BB78','#2B6CB0','#9F7AEA','#ED8936','#F56565']
wedges, texts, autotexts = ax1.pie(
    list(components.values()),
    labels=[f"{k}\n({v/1e6:.1f}M)" for k, v in components.items()],
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.82,
)
for t in texts: t.set_fontsize(8)
for at in autotexts: at.set_fontsize(8); at.set_color('white')
ax1.set_title('Parameter Distribution by Component', fontsize=12, fontweight='bold')

# Bar chart: trainable vs frozen
total_p, trainable_p = count_params(model)
frozen_p = total_p - trainable_p
bars = ax2.bar(['Frozen','Trainable'], [frozen_p/1e6, trainable_p/1e6],
               color=['#4A5568','#48BB78'], edgecolor='white', width=0.5)
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{bar.get_height():.1f}M', ha='center', fontsize=12, fontweight='bold')
ax2.set_ylabel('Parameters (millions)')
ax2.set_title(f'Frozen vs Trainable\n({100*trainable_p/total_p:.1f}% trainable)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, max(frozen_p, trainable_p)/1e6 * 1.25)

plt.suptitle(f'Model Parameter Breakdown  |  Total: {total_p/1e6:.1f}M params', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'model_params.png')
plt.show()
print(f'Total: {total_p/1e6:.1f}M | Trainable: {trainable_p/1e6:.1f}M ({100*trainable_p/total_p:.1f}%)')

In [ ]:
# ── 3C. ViT Patch Tokenization Visualization ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
img_arr = np.array(Image.open(sample_img_path).convert('RGB').resize((224, 224)))

# Original
axes[0].imshow(img_arr)
axes[0].set_title('Original (224×224)', fontsize=11)
axes[0].axis('off')

# Patch grid overlay
axes[1].imshow(img_arr)
patch_size = 16
for x in range(0, 224, patch_size):
    axes[1].axvline(x, color='cyan', linewidth=0.5, alpha=0.7)
for y in range(0, 224, patch_size):
    axes[1].axhline(y, color='cyan', linewidth=0.5, alpha=0.7)
axes[1].set_title('Patch Grid (16×16 patches → 196 tokens)', fontsize=11)
axes[1].axis('off')

# Patch heatmap (mean brightness per patch)
patch_grid = img_arr.reshape(14, 16, 14, 16, 3).mean(axis=(1, 3, 4))
im = axes[2].imshow(patch_grid, cmap='viridis', aspect='auto')
axes[2].set_title('Per-Patch Mean Brightness (14×14)', fontsize=11)
axes[2].set_xlabel('Patch column'); axes[2].set_ylabel('Patch row')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle('ViT-B/16 Patch Tokenization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'vit_patch_tokenization.png')
plt.show()

In [ ]:
class CutOut:
    def __init__(self, num_holes=1, max_h=40, max_w=40):
        self.num_holes=num_holes; self.max_h=max_h; self.max_w=max_w
    def __call__(self, img):
        _, h, w = img.shape; mask = torch.ones_like(img)
        for _ in range(self.num_holes):
            y=random.randint(0,h-1); x=random.randint(0,w-1)
            mask[:, max(0,y-self.max_h//2):min(h,y+self.max_h//2),
                    max(0,x-self.max_w//2):min(w,x+self.max_w//2)] = 0
        return img*mask

def get_train_tf():
    return T.Compose([T.RandomResizedCrop(224,scale=(0.7,1.0),ratio=(0.75,1.33)),
                      T.RandomHorizontalFlip(0.5), T.ColorJitter(0.3,0.3,0.3,0.1),
                      T.RandomGrayscale(0.05), T.ToTensor(), CutOut(1,32,32),
                      T.Normalize(CLIP_MEAN,CLIP_STD)])

def get_eval_tf():
    return T.Compose([T.Resize(256), T.CenterCrop(224),
                      T.ToTensor(), T.Normalize(CLIP_MEAN,CLIP_STD)])

class MixUpCollator:
    def __init__(self,alpha=0.2): self.dist=torch.distributions.Beta(alpha,alpha)
    def __call__(self,batch):
        imgs,labs=zip(*batch); imgs=torch.stack(imgs); labs=torch.tensor(labs,dtype=torch.long)
        lam=self.dist.sample().item(); idx=torch.randperm(imgs.size(0))
        return lam*imgs+(1-lam)*imgs[idx], labs, labs[idx], lam


# DeepFashion2Dataset
# KEY FIX: build ONE filename set via glob() instead of calling p.exists() per row.
# 312K individual exists() calls on Kaggle NFS storage = 20-40 min hang.
# One glob + set lookup = ~5 seconds total.
class DeepFashion2Dataset(Dataset):
    def __init__(self, root, split='train', transform=None, max_samples=None):
        self.transform = transform
        self.samples   = []

        csv_path = DF2_CSV_ROOT / f'{split}.csv'
        if not csv_path.exists():
            alt = DF2_CSV_ROOT / f'{"val" if split=="validation" else split}.csv'
            csv_path = alt if alt.exists() else csv_path

        df = pd.read_csv(csv_path, usecols=['path', 'category_id'])

        img_dir = DF2_IMG_ROOT / split / 'image'
        if not img_dir.exists():
            img_dir = DF2_IMG_ROOT / ('val' if split=='validation' else split) / 'image'
        print(f'DF2 [{split}] img_dir : {img_dir}  exists={img_dir.exists()}')

        # ONE glob to build filename set — avoids 312K individual network exists() calls
        print(f'DF2 [{split}] building filename index from disk...')
        available = {p.name for p in img_dir.glob('*.jpg')}
        print(f'DF2 [{split}] {len(available):,} files on disk')

        skipped = 0
        for row in df.itertuples(index=False):
            if max_samples and len(self.samples) >= max_samples: break
            idx = DF2_ID_TO_IDX.get(int(row.category_id))
            if idx is None: continue
            fname = Path(str(row.path)).name   # "039350.jpg" from full abs path
            if fname in available:
                self.samples.append((img_dir / fname, idx))
            else:
                skipped += 1

        print(f'DF2 [{split}]: {len(self.samples):,} loaded, {skipped:,} skipped')
        if len(self.samples) == 0:
            sample_fname = Path(df['path'].iloc[0]).name
            raise RuntimeError(
                f'0 samples loaded.\n'
                f'img_dir    = {img_dir}\n'
                f'First CSV path = {df["path"].iloc[0]}\n'
                f'Expected fname = {sample_fname}\n'
                f'Disk sample    = {next(iter(available), "NONE")}'
            )

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return (self.transform(Image.open(p).convert('RGB')) if self.transform
                else Image.open(p).convert('RGB')), l


# SyntheticCompatibilityDataset
_UPPER={0,1,2,3,4,5}; _LOWER={6,7,8}; _FULL={9,10,11,12}
def _grp(label): return 'upper' if label in _UPPER else ('lower' if label in _LOWER else 'full')

class SyntheticCompatibilityDataset(Dataset):
    def __init__(self, df2_dataset, transform=None, max_samples=None):
        self.transform=transform
        self._by_grp={'upper':[],'lower':[],'full':[]}
        for path,label in df2_dataset.samples: self._by_grp[_grp(label)].append(path)
        self.anchors=([(p,'upper') for p in self._by_grp['upper']]+
                      [(p,'lower') for p in self._by_grp['lower']])
        if max_samples: self.anchors=self.anchors[:max_samples]
        print(f"Synthetic: {len(self.anchors):,} anchors | upper={len(self._by_grp['upper']):,} lower={len(self._by_grp['lower']):,}")
    def _load(self,p): return Image.open(p).convert('RGB')
    def _rand(self,g): return random.choice(self._by_grp[g])
    def __len__(self): return len(self.anchors)
    def __getitem__(self,i):
        anc_p,anc_g=self.anchors[i]
        pos_p=self._rand('lower' if anc_g=='upper' else 'upper')
        neg_g=random.choice([anc_g,'full']); neg_p=self._rand(neg_g)
        while neg_p==anc_p and len(self._by_grp[neg_g])>1: neg_p=self._rand(neg_g)
        imgs=[self._load(anc_p),self._load(pos_p),self._load(neg_p)]
        if self.transform: imgs=[self.transform(x) for x in imgs]
        return tuple(imgs)


# PolyvoreDataset (optional)
class PolyvoreDataset(Dataset):
    def __init__(self,root,split='train',transform=None,max_samples=None):
        self.root=Path(root); self.transform=transform
        for fname in [f'{split}_no_dup.json',f'{split}.json','train_compatibility.json']:
            jf=self.root/fname
            if jf.exists(): break
        with open(jf) as f: outfits=json.load(f)
        self.outfits=[]; self.all_items=[]
        for o in outfits:
            sid=str(o.get('set_id',o.get('id',''))); items=[str(i) for i in o.get('items',o.get('item_ids',[]))]
            if len(items)>=2: self.outfits.append((sid,items))
            for iid in items: self.all_items.append((sid,iid))
        if max_samples: self.outfits=self.outfits[:max_samples]
        print(f'Polyvore [{split}]: {len(self.outfits):,} outfits')
    def _load(self,sid,iid):
        for ext in ['.jpg','.png','.jpeg']:
            p=self.root/'images'/sid/f'{iid}{ext}'
            if p.exists(): return Image.open(p).convert('RGB')
        raise FileNotFoundError(f'No image: set={sid} item={iid}')
    def __len__(self): return len(self.outfits)
    def __getitem__(self,i):
        sid,items=self.outfits[i]; a,p=random.sample(items,2)
        ns,ni=self.all_items[random.randint(0,len(self.all_items)-1)]
        while ns==sid: ns,ni=self.all_items[random.randint(0,len(self.all_items)-1)]
        imgs=[self._load(sid,a),self._load(sid,p),self._load(ns,ni)]
        if self.transform: imgs=[self.transform(x) for x in imgs]
        return tuple(imgs)


# Build classification loaders
train_tf=get_train_tf(); eval_tf=get_eval_tf()
df2_train=DeepFashion2Dataset(DF2_ROOT,'train',     train_tf,CFG['df2_max_train'])
df2_val  =DeepFashion2Dataset(DF2_ROOT,'validation',eval_tf)
mixup=MixUpCollator(CFG['mixup_alpha'])
train_loader=DataLoader(df2_train,CFG['bs_cls'],  shuffle=True, num_workers=CFG['num_workers'],pin_memory=True,collate_fn=mixup,drop_last=True)
val_loader  =DataLoader(df2_val,  CFG['bs_cls']*2,shuffle=False,num_workers=CFG['num_workers'],pin_memory=True)
print(f'Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}')


## 5. Phase 1 — Classification Training

In [ ]:
def mixup_ce(logits,la,lb=None,lam=1.0):
    if lb is None or lam==1.0: return F.cross_entropy(logits,la)
    return (lam*F.cross_entropy(logits,la,reduction='none')+(1-lam)*F.cross_entropy(logits,lb,reduction='none')).mean()

scaler = GradScaler()

# LLRD: each ViT layer gets a different LR — deeper layers train faster
param_groups = model.visual_encoder.get_llrd_param_groups(
    base_lr=CFG['lr_cls'], decay_rate=CFG['lr_decay']
)
optimizer = AdamW(param_groups, weight_decay=CFG['weight_decay'])
lrs = [g['lr'] for g in param_groups]
print(f'LLRD param groups: {len(param_groups)} | head LR={lrs[0]:.1e} | embed LR={lrs[-1]:.2e}')

# Warmup + cosine: linear warmup for first N epochs, then cosine decay
def _warmup_cosine(epoch):
    we = CFG['warmup_epochs']
    if epoch < we: return (epoch + 1) / max(we, 1)
    p = (epoch - we) / max(CFG['epochs_cls'] - we, 1)
    return 0.5 * (1.0 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _warmup_cosine)

history = defaultdict(list)
best_val_acc = 0.0

def run_cls_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss=correct=total=0
    with (torch.enable_grad() if train else torch.no_grad()):
        for step,batch in enumerate(loader,1):
            if train:
                imgs,la,lb,lam=batch
                imgs=imgs.to(DEVICE); la=la.to(DEVICE); lb=lb.to(DEVICE)
            else:
                imgs,la=batch[0].to(DEVICE),batch[1].to(DEVICE); lb,lam=None,1.0
            if train: optimizer.zero_grad()
            with autocast():
                _,logits=model.forward_classification(imgs)
                loss=mixup_ce(logits,la,lb,lam)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(),CFG['grad_clip'])
                scaler.step(optimizer); scaler.update()
            tot_loss+=loss.item()
            correct+=(logits.argmax(-1)==la).sum().item(); total+=la.size(0)
            if train and step%CFG['log_every']==0:
                print(f'  [{step}/{len(loader)}] loss={loss.item():.4f} acc={correct/total:.4f}')
    return tot_loss/len(loader), correct/total

print('=== Phase 1: Classification Training ===')
t0=time.time()
for epoch in range(1,CFG['epochs_cls']+1):
    tl,ta=run_cls_epoch(train_loader,True)
    vl,va=run_cls_epoch(val_loader,False)
    scheduler.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    history['lr'].append(scheduler.get_last_lr()[0])
    elapsed=time.time()-t0
    print(f'Ep {epoch:02d}/{CFG["epochs_cls"]} | '
          f'train loss={tl:.4f} acc={ta:.4f} | val loss={vl:.4f} acc={va:.4f} | '
          f'lr={history["lr"][-1]:.2e} | {elapsed/60:.1f}min')
    if va>best_val_acc:
        best_val_acc=va
        torch.save(model.state_dict(), CKPT_DIR/'best_classifier.pt')
        print(f'  ✓ New best val acc: {va:.4f}')

torch.save(model.state_dict(), CKPT_DIR/'final_classifier.pt')
print(f'Phase 1 complete. Best val acc: {best_val_acc:.4f}')

## 6. Phase 1 — Comprehensive Evaluation Visualizations

In [ ]:
# ── 6A. Training Curves (4-panel) ────────────────────────────────────────────
eps = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Loss
ax = axes[0,0]
ax.plot(eps, history['train_loss'], 'o-', color='steelblue', label='Train', linewidth=2)
ax.plot(eps, history['val_loss'],   's-', color='coral',     label='Val',   linewidth=2)
ax.fill_between(eps, history['train_loss'], history['val_loss'], alpha=0.1, color='gray')
ax.set_title('Loss per Epoch', fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend()

# Accuracy
ax = axes[0,1]
ax.plot(eps, history['train_acc'], 'o-', color='steelblue', label='Train', linewidth=2)
ax.plot(eps, history['val_acc'],   's-', color='coral',     label='Val',   linewidth=2)
ax.axhline(0.88, color='green', linestyle='--', linewidth=1.5, label='Target 88%')
ax.fill_between(eps, history['val_acc'], 0.88, where=[v<0.88 for v in history['val_acc']],
                alpha=0.15, color='red', label='Gap to target')
ax.set_title('Accuracy per Epoch', fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1); ax.legend()

# Learning rate
ax = axes[1,0]
ax.semilogy(eps, history['lr'], 'o-', color='mediumpurple', linewidth=2)
ax.fill_between(eps, history['lr'], alpha=0.2, color='mediumpurple')
ax.set_title('Learning Rate (log scale)', fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('LR')

# Train-val gap (overfitting monitor)
ax = axes[1,1]
gap = [ta-va for ta,va in zip(history['train_acc'],history['val_acc'])]
colors_gap = ['red' if g>0.05 else 'green' for g in gap]
ax.bar(eps, gap, color=colors_gap, alpha=0.8, edgecolor='white')
ax.axhline(0.05, color='orange', linestyle='--', label='Overfitting threshold (5%)')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Train−Val Accuracy Gap (Overfitting Monitor)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Gap'); ax.legend()

plt.suptitle('Phase 1 — Classification Training Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'training_dashboard.png')
plt.show()

In [ ]:
# ── 6B. Confusion Matrix (best checkpoint) ───────────────────────────────────
model.load_state_dict(torch.load(CKPT_DIR/'best_classifier.pt', map_location=DEVICE))
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs,labs in val_loader:
        imgs=imgs.to(DEVICE)
        _,logits=model.forward_classification(imgs)
        probs=torch.softmax(logits,-1)
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(labs.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds=np.array(all_preds); all_labels=np.array(all_labels); all_probs=np.array(all_probs)
SHORT_CATS = [c.split()[0]+('\n'+' '.join(c.split()[1:])) if len(c.split())>2 else c for c in DF2_CATS]

cm_val = confusion_matrix(all_labels, all_preds)
cm_norm = cm_val.astype(float) / cm_val.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
for ax, mat, title, fmt in zip(axes, [cm_val, cm_norm],
                                ['Raw Counts', 'Row-Normalized (Recall per Class)'],
                                ['d', '.2f']):
    sns.heatmap(mat, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=SHORT_CATS, yticklabels=SHORT_CATS,
                linewidths=0.3, cbar_kws={'shrink':0.8})
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Validation Set Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'confusion_matrix.png', bbox_inches='tight')
plt.show()
print(classification_report(all_labels, all_preds, target_names=DF2_CATS, digits=3))

In [ ]:
# ── 6C. Per-Class Precision / Recall / F1 Bar Charts ─────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score
prec = precision_score(all_labels, all_preds, average=None)
rec  = recall_score(all_labels,    all_preds, average=None)
f1   = f1_score(all_labels,        all_preds, average=None)

x = np.arange(13); w = 0.25
fig, ax = plt.subplots(figsize=(20, 6))
ax.bar(x-w,   prec, w, label='Precision', color='steelblue',   alpha=0.85, edgecolor='white')
ax.bar(x,     rec,  w, label='Recall',    color='coral',        alpha=0.85, edgecolor='white')
ax.bar(x+w,   f1,   w, label='F1',        color='mediumpurple', alpha=0.85, edgecolor='white')
ax.axhline(0.88, color='green', linestyle='--', linewidth=1.5, label='Target 88%')
ax.set_xticks(x); ax.set_xticklabels(DF2_CATS, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score'); ax.legend(fontsize=10)
ax.set_title('Per-Class Precision / Recall / F1 — Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'per_class_metrics.png')
plt.show()

In [ ]:
# ── 6D. ROC Curves (One-vs-Rest) + AUC ───────────────────────────────────────
y_bin = label_binarize(all_labels, classes=range(13))
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
axes = axes.flat
auc_scores = []

for i, (ax, cat) in enumerate(zip(axes, DF2_CATS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores.append(roc_auc)
    ax.plot(fpr, tpr, color=CAT_COLORS[i], lw=2, label=f'AUC={roc_auc:.3f}')
    ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.15, color=CAT_COLORS[i])
    ax.set_title(cat, fontsize=8, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlabel('FPR',fontsize=7); ax.set_ylabel('TPR',fontsize=7)
    ax.tick_params(labelsize=6)

for j in range(len(DF2_CATS), len(list(axes))):
    list(axes)[j].axis('off')

plt.suptitle(f'ROC Curves per Category (One-vs-Rest) | Mean AUC={np.mean(auc_scores):.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'roc_curves.png')
plt.show()
print(f'Mean AUROC: {np.mean(auc_scores):.4f}  (target ≥ 0.91)')
for cat, sc in zip(DF2_CATS, auc_scores):
    print(f'  {cat:<30} AUC={sc:.4f}')

In [ ]:
# ── 6E. t-SNE of Learned Embeddings ──────────────────────────────────────────
print('Extracting embeddings for t-SNE (this takes ~2 min)...')
model.eval()
embs_list, labs_list = [], []
with torch.no_grad():
    for imgs,labs in val_loader:
        imgs=imgs.to(DEVICE)
        emb,_=model.forward_classification(imgs)
        embs_list.append(emb.cpu().numpy())
        labs_list.extend(labs.numpy())
        if len(labs_list)>=3000: break  # 3K samples is enough for t-SNE

embs_arr = np.vstack(embs_list)[:3000]
labs_arr = np.array(labs_list)[:3000]

tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=SEED, verbose=1)
embs_2d = tsne.fit_transform(embs_arr)

fig, ax = plt.subplots(figsize=(14, 11))
for i, cat in enumerate(DF2_CATS):
    mask = labs_arr == i
    ax.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
               color=CAT_COLORS[i], label=cat, alpha=0.6, s=18, edgecolors='none')

ax.legend(fontsize=8, markerscale=1.5, loc='upper right',
          framealpha=0.9, ncol=2)
ax.set_title('t-SNE of Fine-tuned CLIP Embeddings (Validation Set, n=3,000)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(VIZ_DIR/'tsne_embeddings.png', dpi=150)
plt.show()

In [ ]:
# ── 6F. Confidence Distribution per Class ────────────────────────────────────
# How confident is the model on correct vs incorrect predictions?
correct_conf = all_probs[np.arange(len(all_labels)), all_labels][all_preds == all_labels]
wrong_conf   = all_probs[np.arange(len(all_labels)), all_preds][all_preds != all_labels]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(correct_conf, bins=50, color='#48BB78', alpha=0.85, edgecolor='white', label='Correct')
axes[0].hist(wrong_conf,   bins=50, color='#F56565', alpha=0.75, edgecolor='white', label='Wrong')
axes[0].axvline(correct_conf.mean(), color='green', linestyle='--',
                label=f'Correct mean={correct_conf.mean():.3f}')
axes[0].axvline(wrong_conf.mean(),   color='red',   linestyle='--',
                label=f'Wrong mean={wrong_conf.mean():.3f}')
axes[0].set_xlabel('Max Softmax Confidence'); axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence: Correct vs Wrong', fontweight='bold')
axes[0].legend()

# Per-class mean confidence
per_class_conf = [all_probs[all_labels==i, i].mean() for i in range(13)]
bars = axes[1].bar(range(13), per_class_conf, color=CAT_COLORS, edgecolor='white', alpha=0.9)
axes[1].set_xticks(range(13)); axes[1].set_xticklabels(DF2_CATS, rotation=35, ha='right', fontsize=8)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.6)
axes[1].set_ylabel('Mean Confidence (on true class)')
axes[1].set_title('Per-Class Mean Confidence (True Label Score)', fontweight='bold')
axes[1].set_ylim(0, 1)
for bar, v in zip(bars, per_class_conf):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{v:.2f}', ha='center', fontsize=7)

plt.suptitle('Model Confidence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'confidence_analysis.png')
plt.show()

In [ ]:
# ── 6G. Worst Predictions Gallery ────────────────────────────────────────────
# Show images where the model was most wrong (high confidence, wrong class)
wrong_mask = all_preds != all_labels
wrong_confs = all_probs[wrong_mask].max(axis=1)
wrong_idx   = np.where(wrong_mask)[0]
top_wrong   = wrong_idx[np.argsort(wrong_confs)[::-1]][:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
df2_val_paths = [s[0] for s in df2_val.samples]

for ax, idx in zip(axes.flat, top_wrong):
    if idx >= len(df2_val_paths): continue
    img = Image.open(df2_val_paths[idx]).convert('RGB')
    img.thumbnail((200, 250))
    ax.imshow(img)
    true_c  = DF2_CATS[all_labels[idx]]
    pred_c  = DF2_CATS[all_preds[idx]]
    conf    = all_probs[idx, all_preds[idx]]
    ax.set_title(f'True:  {true_c}\nPred: {pred_c}\nConf: {conf:.3f}',
                 fontsize=8, color='white',
                 bbox=dict(boxstyle='round', facecolor='#C53030', alpha=0.85))
    ax.axis('off')

plt.suptitle('Worst Predictions — High Confidence Errors', fontsize=14, fontweight='bold', color='darkred')
plt.tight_layout()
plt.savefig(VIZ_DIR/'worst_predictions.png', bbox_inches='tight')
plt.show()

## 7. Phase 2 — Compatibility Training

In [ ]:
# ── Auto-select compatibility dataset ────────────────────────────────────────
print(f'Compatibility mode: [{COMPAT_MODE.upper()}]')

if COMPAT_MODE == 'polyvore':
    c_train_ds = PolyvoreDataset(POLYVORE_ROOT, 'train', train_tf)
    c_val_ds   = PolyvoreDataset(POLYVORE_ROOT, 'test',  eval_tf)
else:
    # Re-load DeepFashion2 without transforms so SyntheticDataset can read raw paths
    df2_train_raw = DeepFashion2Dataset(DF2_ROOT, 'train',      transform=None)
    df2_val_raw   = DeepFashion2Dataset(DF2_ROOT, 'validation', transform=None)
    c_train_ds = SyntheticCompatibilityDataset(df2_train_raw, train_tf)
    c_val_ds   = SyntheticCompatibilityDataset(df2_val_raw,   eval_tf)

c_train_loader = DataLoader(c_train_ds, CFG['bs_compat'], shuffle=True,
                             num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
c_val_loader   = DataLoader(c_val_ds,   CFG['bs_compat'], shuffle=False,
                             num_workers=CFG['num_workers'], pin_memory=True)
print(f'Compat train batches: {len(c_train_loader):,} | val batches: {len(c_val_loader):,}')

# ── Freeze ViT backbone; only train compatibility head ────────────────────────
for p in model.visual_encoder.vision_model.parameters(): p.requires_grad=False

triplet_loss_fn = nn.TripletMarginLoss(margin=CFG['triplet_margin'])
c_optimizer = AdamW(filter(lambda p:p.requires_grad, model.parameters()),
                    lr=CFG['lr_compat'], weight_decay=CFG['weight_decay'])
c_scheduler = CosineAnnealingLR(c_optimizer, T_max=CFG['epochs_compat'])
c_history = defaultdict(list)
best_c_loss = float('inf')

def run_compat_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot = 0
    with (torch.enable_grad() if train else torch.no_grad()):
        for step,(a,p,n) in enumerate(loader, 1):
            a,p,n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
            if train: c_optimizer.zero_grad()
            with autocast():
                pa,pp,pn = model.forward_triplet(a,p,n)
                loss = triplet_loss_fn(pa,pp,pn)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(c_optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(c_optimizer); scaler.update()
            tot += loss.item()
            if train and step % CFG['log_every'] == 0:
                print(f'  [{step}/{len(loader)}] triplet={loss.item():.4f}')
    return tot / len(loader)

print(f'=== Phase 2: Compatibility Training [{COMPAT_MODE.upper()}] ===')
for epoch in range(1, CFG['epochs_compat']+1):
    tl = run_compat_epoch(c_train_loader, True)
    vl = run_compat_epoch(c_val_loader,   False)
    c_scheduler.step()
    c_history['train'].append(tl); c_history['val'].append(vl)
    print(f'Ep {epoch:02d}/{CFG["epochs_compat"]} | train={tl:.4f} | val={vl:.4f}')
    if vl < best_c_loss:
        best_c_loss = vl
        torch.save(model.state_dict(), CKPT_DIR/'best_compatibility.pt')
        print(f'  ✓ Saved best compat model')

torch.save(model.state_dict(), CKPT_DIR/'final_model.pt')
print('Phase 2 complete.')

In [ ]:
# ── 7A. Compatibility Training Curves + Loss Improvement ─────────────────────
eps_c = range(1, len(c_history['train'])+1)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(eps_c, c_history['train'],'o-',color='steelblue',label='Train',lw=2)
axes[0].plot(eps_c, c_history['val'],  's-',color='coral',    label='Val',  lw=2)
axes[0].fill_between(eps_c,c_history['train'],c_history['val'],alpha=0.1,color='gray')
axes[0].set_title('Triplet Loss per Epoch',fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Triplet Loss'); axes[0].legend()

improvement = [(c_history['val'][0]-v)/c_history['val'][0]*100 for v in c_history['val']]
axes[1].bar(eps_c, improvement, color='#48BB78', alpha=0.85, edgecolor='white')
axes[1].set_title('Val Loss Improvement vs Epoch 1 (%)',fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Improvement (%)')

# Compatibility score distribution on a sample batch
model.eval()
compat_scores_pos, compat_scores_neg = [], []
with torch.no_grad():
    for a,p,n in c_val_loader:
        a,p,n=a.to(DEVICE),p.to(DEVICE),n.to(DEVICE)
        ea,_=model.forward_classification(a)
        ep,_=model.forward_classification(p)
        en,_=model.forward_classification(n)
        compat_scores_pos.extend(model.compatibility.score(ea,ep).cpu().numpy())
        compat_scores_neg.extend(model.compatibility.score(ea,en).cpu().numpy())
        if len(compat_scores_pos)>500: break

axes[2].hist(compat_scores_pos[:500],bins=40,color='#48BB78',alpha=0.75,label='Compatible (same outfit)',edgecolor='white')
axes[2].hist(compat_scores_neg[:500],bins=40,color='#F56565',alpha=0.75,label='Incompatible (diff outfit)',edgecolor='white')
axes[2].axvline(50,color='gray',linestyle='--',label='Score=50 midpoint')
axes[2].set_title('Compatibility Score Distribution',fontweight='bold')
axes[2].set_xlabel('Compatibility Score (0–100)'); axes[2].legend(fontsize=9)

plt.suptitle('Phase 2 — Compatibility Training Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'compatibility_training.png')
plt.show()

In [ ]:
# ── 7B. Compatible vs Incompatible Pairs Gallery ─────────────────────────────
# Works with both Polyvore and Synthetic modes — decodes tensors back to PIL
# so we never need to re-open files or know which dataset was used.
def tensor_to_pil(t):
    mean = torch.tensor(CLIP_MEAN).view(3,1,1)
    std  = torch.tensor(CLIP_STD).view(3,1,1)
    img  = (t.cpu() * std + mean).clamp(0,1)
    return Image.fromarray((img.permute(1,2,0).numpy() * 255).astype(np.uint8))

model.eval()
gallery = []
with torch.no_grad():
    for a_t, p_t, n_t in c_val_loader:
        ea,_ = model.forward_classification(a_t[:4].to(DEVICE))
        ep,_ = model.forward_classification(p_t[:4].to(DEVICE))
        en,_ = model.forward_classification(n_t[:4].to(DEVICE))
        ps   = model.compatibility.score(ea, ep).cpu().numpy()
        ns   = model.compatibility.score(ea, en).cpu().numpy()
        for i in range(min(4, len(ps))):
            gallery.append((
                tensor_to_pil(a_t[i]), tensor_to_pil(p_t[i]), tensor_to_pil(n_t[i]),
                ps[i], ns[i]
            ))
        if len(gallery) >= 4: break

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for row, (img_a, img_p, img_n, pos_s, neg_s) in enumerate(gallery[:4]):
    for col, (img, title, color) in enumerate([
        (img_a, 'Anchor Item',                    '#2D3A4A'),
        (img_p, f'Compatible ✓\nScore: {pos_s:.1f}', '#276749'),
        (img_n, f'Incompatible ✗\nScore: {neg_s:.1f}','#9B2C2C'),
    ]):
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=10, fontweight='bold', color='white',
                                  bbox=dict(boxstyle='round', facecolor=color, alpha=0.85))
        axes[row, col].axis('off')

plt.suptitle(f'Outfit Compatibility Pairs [{COMPAT_MODE.upper()}] — Anchor / Compatible / Incompatible',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR/'compatibility_pairs.png', bbox_inches='tight')
plt.show()

## 8. Grad-CAM Gallery

In [ ]:
class ViTGradCAM:
    def __init__(self,visual_encoder):
        self.enc=visual_encoder; self._acts=self._grads=None
        tgt=self.enc.vision_model.encoder.layers[-1]
        tgt.register_forward_hook(lambda m,i,o: setattr(self,'_acts',(o[0] if isinstance(o,tuple) else o).detach()))
        tgt.register_full_backward_hook(lambda m,gi,go: setattr(self,'_grads',go[0].detach()))
    def generate(self,pv,target_cls=None):
        self.enc.eval()
        pv=pv.clone().requires_grad_(True)
        emb,logits=self.enc(pv)
        if target_cls is None: target_cls=logits.argmax(-1).item()
        self.enc.zero_grad(); logits[0,target_cls].backward()
        acts=self._acts[:,1:,:]; grads=self._grads[:,1:,:]
        cam=F.relu((grads.mean(-1,keepdim=True)*acts).sum(-1)).reshape(1,1,14,14)
        cam=F.interpolate(cam,(224,224),mode='bilinear',align_corners=False).squeeze().cpu().detach().numpy()
        mn,mx=cam.min(),cam.max()
        return (cam-mn)/(mx-mn+1e-8), target_cls

gradcam = ViTGradCAM(model.visual_encoder)

# Sample 1 image per category from val set
cat_samples = {}
for path,lab in df2_val.samples:
    if lab not in cat_samples: cat_samples[lab]=(path,lab)
    if len(cat_samples)==13: break

fig = plt.figure(figsize=(26, 42))
outer = gridspec.GridSpec(13, 1, figure=fig, hspace=0.4)

for row_idx, cls_id in enumerate(range(13)):
    if cls_id not in cat_samples: continue
    path,_ = cat_samples[cls_id]
    orig = Image.open(path).convert('RGB')
    inp  = eval_tf(orig).unsqueeze(0).to(DEVICE)

    # Generate CAMs for top-3 predicted classes
    with torch.no_grad():
        _,logits_t=model.forward_classification(inp)
    top3_classes = logits_t[0].argsort(descending=True)[:3].tolist()

    inner = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[row_idx], wspace=0.05)

    # Col 0: original
    ax0 = fig.add_subplot(inner[0])
    ax0.imshow(orig.resize((224,224)))
    ax0.set_title(f'True:\n{DF2_CATS[cls_id]}', fontsize=7, fontweight='bold',
                  color='white', bbox=dict(facecolor='#2B6CB0',alpha=0.85,boxstyle='round,pad=0.2'))
    ax0.axis('off')

    # Cols 1-3: GradCAM for top-3 classes
    for col_i, tc in enumerate(top3_classes, 1):
        cam,_ = gradcam.generate(inp, target_cls=tc)
        colored = (cm.jet(cam)[:,:,:3]*255).astype(np.uint8)
        overlay = Image.blend(orig.resize((224,224)), Image.fromarray(colored), alpha=0.45)
        ax = fig.add_subplot(inner[col_i])
        ax.imshow(overlay)
        conf = torch.softmax(logits_t[0],-1)[tc].item()
        correct = '✓' if tc==cls_id else '✗'
        fc = '#276749' if tc==cls_id else '#9B2C2C'
        ax.set_title(f'{correct} {DF2_CATS[tc]}\nConf: {conf:.3f}',
                     fontsize=7, color='white', bbox=dict(facecolor=fc,alpha=0.85,boxstyle='round,pad=0.2'))
        ax.axis('off')

    # Col 4: heatmap only (no overlay)
    cam0,_ = gradcam.generate(inp, target_cls=cls_id)
    ax4 = fig.add_subplot(inner[4])
    im4 = ax4.imshow(cam0, cmap='jet'); ax4.axis('off')
    ax4.set_title('Saliency only', fontsize=7)
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

plt.suptitle('Grad-CAM Explanations — All 13 Categories\n'
             '(Col 1: Original | Cols 2-4: Top-3 predicted class heatmaps | Col 5: Pure saliency)',
             fontsize=14, fontweight='bold', y=1.002)
plt.savefig(VIZ_DIR/'gradcam_gallery.png', bbox_inches='tight', dpi=100)
plt.show()
print('Grad-CAM gallery saved.')

## 9. Final Dashboard — All Metrics at a Glance

In [ ]:
# ── Final Metrics Dashboard ───────────────────────────────────────────────────
fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor('#1A1A2E')
gs_main = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

def dark_ax(ax):
    ax.set_facecolor('#2D3A4A')
    ax.tick_params(colors='#CBD5E0'); ax.xaxis.label.set_color('#CBD5E0')
    ax.yaxis.label.set_color('#CBD5E0'); ax.title.set_color('#EDF2F7')
    for spine in ax.spines.values(): spine.set_edgecolor('#4A5568')
    return ax

# KPI Boxes (row 0)
kpis = [
    ('Best Val Accuracy', f'{best_val_acc:.4f}', '0.88 target', best_val_acc>=0.88),
    ('Mean AUROC',        f'{np.mean(auc_scores):.4f}', '0.91 target', np.mean(auc_scores)>=0.91),
    ('Best Triplet Loss', f'{best_c_loss:.4f}', 'Lower is better', True),
    ('Total Parameters',  f'{total_p/1e6:.1f}M', f'{trainable_p/1e6:.1f}M trainable', True),
]
for col, (title, value, sub, ok) in enumerate(kpis):
    ax = fig.add_subplot(gs_main[0, col])
    ax.set_facecolor('#276749' if ok else '#9B2C2C')
    ax.axis('off')
    ax.text(0.5, 0.65, value, ha='center', va='center', fontsize=26,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.30, title, ha='center', va='center', fontsize=11,
            color='#E2E8F0', transform=ax.transAxes)
    ax.text(0.5, 0.10, sub, ha='center', va='center', fontsize=8,
            color='#BEE3F8', transform=ax.transAxes)
    for s in ax.spines.values(): s.set_visible(False)

# Row 1: loss curves, accuracy curves
ax_l = dark_ax(fig.add_subplot(gs_main[1, :2]))
eps_all = range(1, len(history['train_loss'])+1)
ax_l.plot(eps_all, history['train_loss'], 'o-', color='#63B3ED', label='Train Loss', lw=2)
ax_l.plot(eps_all, history['val_loss'],   's-', color='#FC8181', label='Val Loss',   lw=2)
ax_l.set_title('Classification Loss',fontweight='bold'); ax_l.legend(facecolor='#2D3A4A',labelcolor='white')

ax_a = dark_ax(fig.add_subplot(gs_main[1, 2:]))
ax_a.plot(eps_all, history['train_acc'],'o-',color='#63B3ED',label='Train Acc',lw=2)
ax_a.plot(eps_all, history['val_acc'],  's-',color='#FC8181',label='Val Acc',  lw=2)
ax_a.axhline(0.88,color='#68D391',linestyle='--',lw=1.5,label='Target 88%')
ax_a.set_ylim(0,1); ax_a.set_title('Classification Accuracy',fontweight='bold')
ax_a.legend(facecolor='#2D3A4A',labelcolor='white')

# Row 2: per-class F1, AUC bar, compatibility loss
ax_f1 = dark_ax(fig.add_subplot(gs_main[2, :2]))
bars2 = ax_f1.bar(range(13), f1, color=CAT_COLORS, edgecolor='#1A1A2E', alpha=0.9)
ax_f1.axhline(f1.mean(), color='white', linestyle='--', lw=1.5, label=f'Mean F1={f1.mean():.3f}')
ax_f1.set_xticks(range(13))
ax_f1.set_xticklabels([c.split()[0] for c in DF2_CATS], rotation=45, ha='right', fontsize=8)
ax_f1.set_ylim(0,1.1); ax_f1.set_title('Per-Class F1 Score',fontweight='bold')
ax_f1.legend(facecolor='#2D3A4A',labelcolor='white')

ax_auc = dark_ax(fig.add_subplot(gs_main[2, 2]))
bars3 = ax_auc.bar(range(13), auc_scores, color=CAT_COLORS, edgecolor='#1A1A2E', alpha=0.9)
ax_auc.axhline(0.91, color='#68D391', linestyle='--', lw=1.5, label='Target 0.91')
ax_auc.axhline(np.mean(auc_scores),color='white',linestyle=':',lw=1.5,label=f'Mean={np.mean(auc_scores):.3f}')
ax_auc.set_xticks(range(13))
ax_auc.set_xticklabels([c.split()[0] for c in DF2_CATS], rotation=45, ha='right', fontsize=8)
ax_auc.set_ylim(0.5,1.05); ax_auc.set_title('Per-Class AUROC',fontweight='bold')
ax_auc.legend(facecolor='#2D3A4A',labelcolor='white',fontsize=8)

ax_ct = dark_ax(fig.add_subplot(gs_main[2, 3]))
eps_c2=range(1,len(c_history['train'])+1)
ax_ct.plot(eps_c2,c_history['train'],'o-',color='#63B3ED',label='Train',lw=2)
ax_ct.plot(eps_c2,c_history['val'],  's-',color='#FC8181',label='Val',  lw=2)
ax_ct.set_title('Triplet Loss (Compat.)',fontweight='bold')
ax_ct.legend(facecolor='#2D3A4A',labelcolor='white')

plt.suptitle('FashionSense AI — Final Training Dashboard',
             fontsize=18, fontweight='bold', color='white', y=1.01)
plt.savefig(VIZ_DIR/'final_dashboard.png', bbox_inches='tight', facecolor='#1A1A2E', dpi=130)
plt.show()
print('Final dashboard saved.')

In [ ]:
# ── Save all artifacts + config ───────────────────────────────────────────────
cfg_out = {
    'clip_model_name': CFG['clip_model'],
    'num_classes':     CFG['num_classes'],
    'embedding_dim':   CFG['emb_dim'],
    'categories':      DF2_CATS,
    'best_val_acc':    float(best_val_acc),
    'mean_auroc':      float(np.mean(auc_scores)),
    'best_triplet_loss': float(best_c_loss),
    'per_class_f1':    {c: float(v) for c,v in zip(DF2_CATS, f1)},
    'per_class_auc':   {c: float(v) for c,v in zip(DF2_CATS, auc_scores)},
}
with open(CKPT_DIR/'model_config.json','w') as f:
    json.dump(cfg_out, f, indent=2)

print('\n=== Saved Artifacts ===')
for p in sorted(CKPT_DIR.glob('*')):
    print(f'  {p.name:<35}  {p.stat().st_size/1e6:.1f} MB')

print('\n=== Saved Visualizations ===')
for p in sorted(VIZ_DIR.glob('*.png')):
    print(f'  {p.name}')

print(f'\n✓ All done. Download from /kaggle/working/')